# AgenticEd Labs · Week 4 — Multi-Agent Systems: Orchestration, Verification, Human Checkpoints

<a href="https://colab.research.google.com/github/AgenticEdOrg/AgenticEd/blob/main/labs/week4_multi_agent_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

| | |
|---|---|
| **Time** | 90–110 minutes |
| **Approx. API cost** | ≈ US$0.15 with the default low-cost model |
| **Prerequisites** | Weeks 1–3 |
| **You'll use** | LangGraph, LangChain chat models, CrewAI (optional), rank-bm25 |

**Learning objectives**
- Build an **orchestrator–workers** graph in **LangGraph** with typed state and parallel fan-out (`Send`)
- Pass **structured handoffs** between agents and **verify** them against sources
- **Inject a fault** and watch verification catch it before it compounds
- Pause for **human approval** with `interrupt()` and resume from a **checkpoint**
- Build the same team in **CrewAI** and compare the two programming models

> **Before you start:** add `OPENAI_API_KEY` *or* `ANTHROPIC_API_KEY` to Colab Secrets (key icon in the left sidebar → enable notebook access), or to `labs/.env` locally. See `00_environment_check.ipynb` if anything fails.

In [ ]:
# ── Setup (run once) ─────────────────────────────────────────────────────────
# In Colab: clones the course repo and installs the pinned requirements (2–3 minutes the first time).
# If Colab asks you to restart the runtime after installing, do so and run this cell again.
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/AgenticEd").exists():
        !git clone -q --depth 1 https://github.com/AgenticEdOrg/AgenticEd.git /content/AgenticEd
    %cd /content/AgenticEd/labs
    %pip install -q -r requirements.txt
sys.path.insert(0, str(pathlib.Path.cwd()))

import labkit
cfg = labkit.setup()          # reads your key, picks the model, prints what it chose

## The task

Produce a **one-page briefing for new teachers**: what they must know about AI tools and student data at Lakeview. It needs research across several policies, writing, and fact-checking — and a human must approve before it's "published" to staff.

| Pattern | Shape | Use when |
|---|---|---|
| Prompt chaining | A → B → C | fixed stages |
| Routing | classifier → specialist | distinct input types |
| **Orchestrator–workers** | planner fans out to workers, results merged | subtasks not known in advance |
| Evaluator–optimizer | generate ⇄ critique loop | clear quality criteria |

We combine orchestrator–workers with an evaluator loop and a human checkpoint.

In [ ]:
import re, operator, json
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

DOCS = {d["id"]: d for d in labkit.load_corpus()}
SECTIONS = [(d["id"], s.strip()) for d in DOCS.values() for s in re.split(r"\n(?=## )", d["text"]) if s.strip()]
_tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
_bm25 = BM25Okapi([_tok(s) for _, s in SECTIONS])

def retrieve(query, k=4):
    scores = _bm25.get_scores(_tok(query))
    return [{"doc_id": SECTIONS[i][0], "text": SECTIONS[i][1][:900]} for i in sorted(range(len(SECTIONS)), key=lambda i: -scores[i])[:k]]

llm = cfg.langchain()
def structured(schema):
    """Structured output via tool calling — the most portable method across providers."""
    return llm.with_structured_output(schema, method="function_calling") if cfg.provider == "openai" else llm.with_structured_output(schema)

## 1 · Typed handoffs and graph state

In [ ]:
class Plan(BaseModel):
    questions: list[str] = Field(min_length=2, max_length=4, description="Research questions, one per worker")

class Finding(BaseModel):
    claim: str = Field(description="One factual statement for the briefing")
    doc_id: str = Field(description="Source document id")
    quote: str = Field(description="Exact sentence copied from the source that supports the claim")

class Findings(BaseModel):
    findings: list[Finding] = Field(min_length=1, max_length=4)

class Review(BaseModel):
    approved: bool
    problems: list[str] = Field(default_factory=list)

class BriefState(TypedDict, total=False):
    topic: str
    questions: list[str]
    findings: Annotated[list[dict], operator.add]     # reducer: parallel workers append, never overwrite
    draft: str
    review: dict
    revisions: int
    published: bool
    log: Annotated[list[str], operator.add]

## 2 · The agents (graph nodes)

In [ ]:
from langgraph.types import Send, interrupt, Command

INJECT_FAULT = False   # flipped later to simulate a researcher that invents a fact

def planner(state: BriefState):
    plan = structured(Plan).invoke(
        f"Break this briefing topic into 2-4 focused research questions answerable from school district policies.\nTopic: {state['topic']}")
    return {"questions": plan.questions, "log": [f"planner → {len(plan.questions)} questions"]}

def fan_out(state: BriefState):
    return [Send("researcher", {"question": q}) for q in state["questions"]]

def researcher(task: dict):
    q = task["question"]
    sources = retrieve(q)
    out = structured(Findings).invoke(
        "Answer the research question using ONLY these sources. For each finding copy an exact supporting sentence as `quote`.\n"
        f"Question: {q}\nSources:\n" + json.dumps(sources, indent=1))
    findings = [f.model_dump() for f in out.findings]
    if INJECT_FAULT and "data" in q.lower():
        findings.append({"claim": "Teachers may paste student names into any chatbot if the student agrees.",
                         "doc_id": "student-data-privacy", "quote": "Teachers may paste student names into any chatbot if the student agrees."})
    return {"findings": findings, "log": [f"researcher[{q[:40]}…] → {len(findings)} findings"]}

def writer(state: BriefState):
    problems = state.get("review", {}).get("problems", [])
    prompt = ("Write a concise briefing (max 180 words, bullet points) for new teachers using ONLY these findings. "
              "Do not add facts.\nFindings:\n" + json.dumps(state["findings"], indent=1))
    if problems:
        prompt += "\nA reviewer rejected the previous draft. Remove or fix these problems:\n- " + "\n- ".join(problems)
    draft = llm.invoke(prompt).content
    return {"draft": draft, "revisions": state.get("revisions", -1) + 1, "log": ["writer → draft"]}

def verifier(state: BriefState):
    # 1) Deterministic check: every quote must literally appear in its cited document.
    problems, good = [], []
    for f in state["findings"]:
        doc = DOCS.get(f["doc_id"], {}).get("text", "")
        quote_ok = f["quote"].strip(" .\"'")[:60].lower() in doc.lower()
        (good if quote_ok else problems).append(f)
    problem_msgs = [f"Unsupported claim (quote not found in {f['doc_id']}): {f['claim']}" for f in problems]
    # 2) Model check: does the draft only state things supported by the verified findings?
    review = structured(Review).invoke(
        "You are a strict fact-checker. Approve only if every statement in the draft is supported by the verified findings.\n"
        f"Verified findings:\n{json.dumps(good, indent=1)}\n\nDraft:\n{state['draft']}")
    all_problems = problem_msgs + review.problems
    approved = review.approved and not problem_msgs
    return {"review": {"approved": approved, "problems": all_problems},
            "log": [f"verifier → {'approved' if approved else str(len(all_problems)) + ' problem(s)'}"]}

def route_after_review(state: BriefState):
    if state["review"]["approved"]:
        return "human_approval"
    return "writer" if state["revisions"] < 1 else "human_approval"    # one revision, then escalate to a human

def human_approval(state: BriefState):
    decision = interrupt({"draft": state["draft"], "review": state["review"]})   # pauses the graph here
    return {"published": decision == "approve", "log": [f"human → {decision}"]}

Two production details worth noticing:

- **`findings` uses a reducer** (`operator.add`): parallel researchers append to the same list safely.
- **The verifier is partly deterministic.** It checks that each quote literally appears in its source before asking a model anything. Cheap, exact checks first; model judgment second.

## 3 · Wire the graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(BriefState)
for name, fn in [("planner", planner), ("researcher", researcher), ("writer", writer),
                 ("verifier", verifier), ("human_approval", human_approval)]:
    builder.add_node(name, fn)
builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", fan_out, ["researcher"])
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "verifier")
builder.add_conditional_edges("verifier", route_after_review, ["writer", "human_approval"])
builder.add_edge("human_approval", END)

graph = builder.compile(checkpointer=InMemorySaver())     # checkpointing makes pause/resume possible
print(graph.get_graph().draw_mermaid())

## 4 · Run it — and approve at the checkpoint

In [ ]:
import os, uuid

def run_briefing(topic):
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    state = graph.invoke({"topic": topic, "revisions": -1}, config)
    for line in state.get("log", []):
        print(" ·", line)
    if "__interrupt__" in state:
        pending = state["__interrupt__"][0].value
        print("\n─── DRAFT AWAITING APPROVAL ───\n" + pending["draft"])
        print("\nreviewer:", "approved" if pending["review"]["approved"] else pending["review"]["problems"])
        decision = "approve" if os.environ.get("AGENTICED_AUTO_APPROVE") == "1" else input("\napprove / reject: ").strip().lower()
        state = graph.invoke(Command(resume=decision), config)        # resumes from the checkpoint
        print("\npublished:", state.get("published"))
    return state

TOPIC = "What new teachers must know about using AI tools and handling student data at Lakeview"
state = run_briefing(TOPIC)

## 5 · Fault injection: does verification stop error compounding?

Now one researcher invents a "fact" with a fake quote — the classic failure that turns into a confident, published mistake in an unchecked chain.

In [ ]:
INJECT_FAULT = True
faulty_state = run_briefing(TOPIC)
INJECT_FAULT = False
print("\nFinal reviewer problems:", faulty_state.get("review", {}).get("problems"))

The deterministic quote check flags the fabricated finding and the draft goes back for revision; if it's still not clean after one revision, it reaches the human **with the problems attached** instead of being silently published.

## 6 · The same team in CrewAI (optional)

CrewAI organises work as **role-playing agents** with **tasks**; LangGraph as an explicit **state machine**. Same goal, different trade-offs. (CrewAI needs Python ≤ 3.13 — Colab is fine.)

In [ ]:
try:
    from crewai import Agent as CrewAgent, Task, Crew, Process
    from crewai.tools import tool
except ImportError:
    CrewAgent = None
    print("CrewAI not installed — skipping this section.")

if CrewAgent:
    @tool("search_policies")
    def search_policies_tool(query: str) -> str:
        """Search Lakeview district policies and return relevant sections with their doc_id."""
        return json.dumps(retrieve(query), indent=1)

    crew_llm = cfg.crewai()
    researcher_agent = CrewAgent(role="Policy researcher", goal="Find exact, cited facts in district policies",
                                 backstory="You never state anything you can't quote from a policy.",
                                 tools=[search_policies_tool], llm=crew_llm, max_iter=4, verbose=False)
    writer_agent = CrewAgent(role="Staff briefing writer", goal="Write clear, accurate briefings for teachers",
                             backstory="You only use facts the researcher provides.", llm=crew_llm, verbose=False)
    research_task = Task(description="Research: {topic}. Return 4-6 facts, each with its doc_id and an exact quote.",
                         expected_output="A list of facts with doc_id and quote", agent=researcher_agent)
    write_task = Task(description="Write a briefing of at most 180 words in bullet points using only the research.",
                      expected_output="A bullet-point briefing", agent=writer_agent, context=[research_task])
    crew = Crew(agents=[researcher_agent, writer_agent], tasks=[research_task, write_task], process=Process.sequential)
    crew_result = crew.kickoff(inputs={"topic": TOPIC})
    print(crew_result.raw)

| | LangGraph | CrewAI |
|---|---|---|
| Mental model | explicit graph / state machine | team of role-based agents + tasks |
| Control flow | you define every edge and condition | framework orchestrates (sequential / hierarchical) |
| Parallelism | `Send` fan-out with reducers | async tasks, hierarchical manager |
| Human-in-the-loop | `interrupt()` + checkpoint resume | `human_input=True` on tasks |
| Deterministic checks | any node can be plain Python | task guardrails |
| Best when | you need precise control, auditability, resumability | fast prototyping of role-based collaboration |

### Exercise — add a guard node


Add a node `pii_guard` between `writer` and `verifier` that rejects drafts containing an email address (regex) by returning `{"review": {"approved": False, "problems": ["draft contains an email address"]}}`, otherwise passes through (`{}`). Rebuild the graph and run the check.

In [ ]:
# TODO: define pii_guard(state) and rebuild `graph` with writer → pii_guard → verifier

In [ ]:
nodes = set(graph.get_graph().nodes)
print("✓ Exercise passed" if "pii_guard" in nodes else "Not done yet — add a pii_guard node and recompile the graph.")

<details><summary><b>Show a solution</b></summary>

```python
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
def pii_guard(state: BriefState):
    if EMAIL.search(state["draft"]):
        return {"review": {"approved": False, "problems": ["draft contains an email address"]}}
    return {}

builder = StateGraph(BriefState)
for name, fn in [("planner", planner), ("researcher", researcher), ("writer", writer), ("pii_guard", pii_guard),
                 ("verifier", verifier), ("human_approval", human_approval)]:
    builder.add_node(name, fn)
builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", fan_out, ["researcher"])
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "pii_guard")
builder.add_edge("pii_guard", "verifier")
builder.add_conditional_edges("verifier", route_after_review, ["writer", "human_approval"])
builder.add_edge("human_approval", END)
graph = builder.compile(checkpointer=InMemorySaver())
```

</details>

## Governance note — accountability in agent chains

When several agents contribute to an output, responsibility diffuses unless you design against it: **log every handoff** (our `log` field), **keep provenance** (each claim carries `doc_id` and `quote`), and **name a human owner** for the final decision (the `interrupt`). These are exactly the records an auditor — or an EU AI Act conformity assessment for a high-risk system — will ask for.

## Summary

- Orchestrator–workers + verification + human checkpoint is a robust default for multi-step content work.
- Typed handoffs and reducers make parallel agents safe.
- Put cheap deterministic checks before model judgment.
- Checkpoints enable pause, resume, and audit.

**Further reading:** [LangGraph docs](https://langchain-ai.github.io/langgraph/) · [CrewAI docs](https://docs.crewai.com/) · Anthropic, [*How we built our multi-agent research system*](https://www.anthropic.com/engineering/built-multi-agent-research-system)